# Whisper Bengali Fine-tuning on Pre-chunked Dataset

## 1. Installation

In [ ]:
!pip install -q torch torchaudio
!pip install -q transformers>=4.30.0
!pip install -q datasets
!pip install -q soundfile
!pip install -q accelerate
!pip install -q evaluate
!pip install -q jiwer
!pip install -q scikit-learn
!pip install -q tensorboard

In [ ]:
import os
import shutil

# Remove old training outputs to free space
if os.path.exists('./whisper-bengali-finetuned'):
    shutil.rmtree('./whisper-bengali-finetuned')
    print('✅ Cleaned old outputs')

## 2. Imports

In [ ]:
import os
import json
import torch
import numpy as np
import soundfile as sf
import gc
from pathlib import Path
from tqdm.auto import tqdm
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import warnings
warnings.filterwarnings('ignore')

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    TrainerCallback,
)
from datasets import Dataset, Audio
from sklearn.model_selection import train_test_split
import evaluate

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

## 3. Configuration

### 🔴🔴 Change `chunked_dir` path to the actual chunked dataset path

In [ ]:
CONFIG = {
    # Model
    'model_name'  : 'bengaliAI/tugstugi_bengaliai-asr_whisper-medium',
    'language'    : 'bn',
    'task'        : 'transcribe',

    # ── Pre-chunked dataset path ──────────────────────────────────────────
    # Each subfolder (train_001, train_002 ...) contains:
    #   chunk_audio/        *.wav  (≤28s, silence-stripped)
    #   chunk_annotations/  *.txt  (ground-truth aligned text)
    'chunked_dir' : '/kaggle/input/datasets/aurchi/chunked-asr-train-cleaned/chunked_asr_train_1',

    # Audio
    'sampling_rate': 16000,

    # Dataset split
    'train_split' : 0.9,
    'random_seed' : 42,

    # Training hyperparameters ################################################################ CHANGE IF NEEDED
    'num_epochs'                : 5, #20
    'batch_size'                : 16,
    'gradient_accumulation_steps': 2,
    'learning_rate'             : 1e-5,
    'warmup_steps'              : 350,
    'weight_decay'              : 0.01,
    'gradient_checkpointing'    : False,
    'fp16'                      : False,
    'bf16'                      : False,

    # Evaluation ################################################################ CHANGE IF NEEDED
    'eval_steps'           : 129,
    'save_steps'           : 129,
    'logging_steps'        : 100,
    'save_total_limit'     : 1,
    'metric_for_best_model': 'wer',
    'greater_is_better'    : False,

    # Output
    'output_dir': './whisper-bengali-finetuned',
}

print('✅ Configuration loaded')

## 4. Load Dataset from Pre-chunked Directory

In [ ]:
def load_chunked_dataset(chunked_dir):
    """
    Walk chunked_asr_train/ and collect all (wav_path, text) pairs.
    Structure expected:
        chunked_dir/
            train_001/
                chunk_audio/       train_001_chunk_0001.wav ...
                chunk_annotations/ train_001_chunk_0001.txt ...
            train_002/
                ...
    """
    chunked_dir = Path(chunked_dir)
    data = []
    skipped = 0

    file_dirs = sorted([d for d in chunked_dir.iterdir() if d.is_dir()])
    print(f'Found {len(file_dirs)} source audio folders')

    for file_dir in tqdm(file_dirs, desc='Loading chunks'):
        audio_dir = file_dir / 'chunk_audio'
        annot_dir = file_dir / 'chunk_annotations'

        if not audio_dir.exists() or not annot_dir.exists():
            skipped += 1
            continue

        for wav_path in sorted(audio_dir.glob('*.wav')):
            txt_path = annot_dir / (wav_path.stem + '.txt')
            if not txt_path.exists():
                skipped += 1
                continue

            text = txt_path.read_text(encoding='utf-8').strip()
            if not text:
                skipped += 1
                continue

            data.append({
                'audio_path': str(wav_path),
                'text'      : text,
                'file_id'   : wav_path.stem,
            })

    print(f'\n✅ Loaded {len(data)} chunks')
    if skipped:
        print(f'   Skipped {skipped} (missing pair or empty text)')
    return data

raw_data = load_chunked_dataset(CONFIG['chunked_dir'])

# Basic stats
durations = []
for d in raw_data[:200]:   # sample first 200 to estimate
    info = sf.info(d['audio_path'])
    durations.append(info.duration)
avg_dur = sum(durations) / len(durations) if durations else 0
print(f'\nDataset stats (sample of 200):')
print(f'  Total chunks  : {len(raw_data)}')
print(f'  Avg duration  : {avg_dur:.1f}s')
print(f'  Est. total hrs: {len(raw_data) * avg_dur / 3600:.1f}h')

In [ ]:
import re


def clean_bengali_punctuation(text):
    """
    Cleans text by keeping only:
    - Bengali characters (letters, vowels, matras, anusvara, visarga, 
      chandrabindu, hasanta, khanda ta, etc.) [U+0980-U+09FF]
    - ZWJ and ZWNJ for conjunct control [U+200C, U+200D]
    - English digits (0-9)
    - Hyphen (-)
    - Spaces
    Removes everything else including English letters, Bengali/Devanagari
    punctuation (।॥), commas, question marks, etc.
    """
    # Whitelist: Bengali block + ZWJ/ZWNJ + English digits + hyphen + space
    text = re.sub(r'[^\u0980-\u09FF\u200C\u200D0-9 \-]', '', text)
    # Collapse multiple spaces
    text = re.sub(r' +', ' ', text)
    return text.strip()



# Show before/after example
if raw_data:
    sample = raw_data[0]
    print('Before cleaning:')
    print(f'  {sample["text"][:200]}...\n')
    
    cleaned = clean_bengali_punctuation(sample['text'])
    print('After cleaning:')
    print(f'  {cleaned[:200]}...\n')

# Apply to all data
print(f'Cleaning {len(raw_data)} annotations...')
for item in raw_data:
    item['text'] = clean_bengali_punctuation(item['text'])

print('✅ Punctuation removed from all annotations')

## 5. Train / Validation Split

In [ ]:
train_data, val_data = train_test_split(
    raw_data,
    train_size=CONFIG['train_split'],
    random_state=CONFIG['random_seed'],
    shuffle=True,
)

print(f'Train chunks : {len(train_data)}')
print(f'Val chunks   : {len(val_data)}')

## 6. Build HuggingFace Datasets

In [ ]:
def to_hf_dataset(data):
    return Dataset.from_list([
        {'audio': d['audio_path'], 'text': d['text']}
        for d in data
    ])

train_dataset_raw = to_hf_dataset(train_data)
val_dataset_raw   = to_hf_dataset(val_data)

print(f'Train HF dataset: {len(train_dataset_raw)} samples')
print(f'Val HF dataset  : {len(val_dataset_raw)} samples')

## 7. Load Processor and Model

In [ ]:
print('Loading processor...')
processor = WhisperProcessor.from_pretrained(
    CONFIG['model_name'],
    language=CONFIG['language'],
    task=CONFIG['task'],
)

print('Loading model...')
model = WhisperForConditionalGeneration.from_pretrained(
    CONFIG['model_name'],
    torch_dtype=torch.float16 if CONFIG['fp16'] else torch.float32,
)

# Configure for Bengali
model.generation_config.language = CONFIG['language']
model.generation_config.task     = CONFIG['task']
model.config.forced_decoder_ids  = processor.get_decoder_prompt_ids(
    language=CONFIG['language'],
    task=CONFIG['task'],
)
model.config.suppress_tokens = []

# Memory optimisation
if CONFIG['gradient_checkpointing']:
    model.config.use_cache = False
    model.gradient_checkpointing_enable()

print(f'✅ Model loaded ({model.num_parameters()/1e6:.1f}M params)')

## 8. Preprocess Dataset

In [ ]:
def prepare_dataset(batch):
    """
    Convert raw audio + text into Whisper input features and token labels.
    """
    # Load audio with soundfile
    audio, sr = sf.read(batch['audio'])
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    audio = audio.astype(np.float32)

    batch['input_features'] = processor.feature_extractor(
        audio,
        sampling_rate=CONFIG['sampling_rate'],
        return_tensors='np',
    ).input_features[0]

    batch['labels'] = processor.tokenizer(batch['text']).input_ids
    return batch
    

print('Preprocessing training data...')
train_dataset = train_dataset_raw.map(
    prepare_dataset,
    remove_columns=train_dataset_raw.column_names,
    desc='Train',
)

print('Preprocessing validation data...')
val_dataset = val_dataset_raw.map(
    prepare_dataset,
    remove_columns=val_dataset_raw.column_names,
    desc='Val',
)

print(f'\n✅ Preprocessing done')
print(f'   Train: {len(train_dataset)} samples')
print(f'   Val  : {len(val_dataset)} samples')

In [ ]:
train_dataset = train_dataset.filter(lambda x: len(x['labels']) <= 444)
val_dataset   = val_dataset.filter(lambda x: len(x['labels']) <= 444)

In [ ]:
# Calculate steps per epoch and update eval/save steps
steps_per_epoch = len(train_dataset) // (CONFIG['batch_size'] * CONFIG['gradient_accumulation_steps'])
if len(train_dataset) % (CONFIG['batch_size'] * CONFIG['gradient_accumulation_steps']) > 0:
    steps_per_epoch += 1

print(f'📊 Training Dataset Analysis:')
print(f'   Total samples     : {len(train_dataset)}')
print(f'   Batch size        : {CONFIG["batch_size"]}')
print(f'   Grad accumulation : {CONFIG["gradient_accumulation_steps"]}')
print(f'   Effective batch   : {CONFIG["batch_size"] * CONFIG["gradient_accumulation_steps"]}')
print(f'   Steps per epoch   : {steps_per_epoch}')

# Update CONFIG to match steps per epoch
CONFIG['eval_steps'] = steps_per_epoch
CONFIG['save_steps'] = steps_per_epoch

print(f'\n✅ Updated eval_steps and save_steps to {steps_per_epoch}')

## 9. Data Collator

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{'input_features': f['input_features']} for f in features]
        label_features = [{'input_ids'     : f['labels']}         for f in features]

        batch = self.processor.feature_extractor.pad(input_features, return_tensors='pt')
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors='pt')

        labels = labels_batch['input_ids'].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        # Strip leading BOS token if present (Whisper adds it during generate)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch['labels'] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print('✅ Data collator ready')

## 10. WER Evaluation Callback (Chunk-level)

In [ ]:
import unicodedata

ZW = r"[\u200B-\u200D\uFEFF]"  # zero-width space/joiners + BOM

def normalize_bn_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    s = unicodedata.normalize("NFC", s)
    s = re.sub(ZW, "", s)
    s = s.replace("\u00A0", " ")
    s = " ".join(s.split())
    return s


class ChunkWERCallback(TrainerCallback):
    """
    Evaluates WER directly on validation chunks.
    Much faster than the VAD+inference pipeline approach since
    chunks are already ≤28s and paired with ground-truth text.
    """
    def __init__(self, val_data, processor):
        self.val_data   = val_data
        self.processor  = processor
        self.best_wer   = float('inf')
        self.patience_counter = 0 
        self.last_val_loss = float('inf')
        self.wer_metric = evaluate.load('wer')

    def on_evaluate(self, args, state, control, model, **kwargs):
        from transformers import GenerationConfig
        model.generation_config = GenerationConfig.from_pretrained("openai/whisper-medium")
        model.generation_config.language = CONFIG['language']
        model.generation_config.task = CONFIG['task']
        
        print('\n' + '='*60)
        print('CHUNK-LEVEL WER EVALUATION')
        print('='*60)

        model.eval()
        predictions = []
        references  = []

        for sample in tqdm(self.val_data, desc='Evaluating'):
            audio, sr = sf.read(sample['audio_path'])
            if audio.ndim > 1:
                audio = audio.mean(axis=1)
            audio = audio.astype(np.float32)

            inputs = self.processor.feature_extractor(
                audio,
                sampling_rate=CONFIG['sampling_rate'],
                return_tensors='pt',
            ).input_features.to(device)

            if CONFIG['fp16']:
                inputs = inputs.half()

            with torch.no_grad():
                predicted_ids = model.generate(
                    inputs,
                    language=CONFIG['language'],
                    task=CONFIG['task'],
                    max_new_tokens=440,
                )

            pred_text = self.processor.tokenizer.batch_decode(
                predicted_ids, skip_special_tokens=True
            )[0].strip()

            # Normalize both prediction and reference to handle Unicode inconsistencies
            pred_text_normalized = normalize_bn_text(pred_text)
            ref_text_normalized = normalize_bn_text(sample['text'])

            predictions.append(pred_text_normalized)
            references.append(ref_text_normalized)

        wer = self.wer_metric.compute(
            predictions=predictions, references=references
        )
        print(f'\n📊 Validation WER (chunk-level): {wer:.4f}')

        if wer < self.best_wer - 0.001:
            self.best_wer = wer
            self.patience_counter = 0
            print(f'✨ New best WER: {self.best_wer:.4f} at step {state.global_step}')
            best_path = os.path.join(args.output_dir, 'best_model')
            model.save_pretrained(best_path)
            self.processor.save_pretrained(best_path)
            print(f'   Saved to: {best_path}')
        else:
            self.patience_counter += 1
            print(f'⚠️ No improvement. Patience: {self.patience_counter}/2')
            if self.patience_counter >= 2:
                print('🛑 Early stopping triggered!')
                control.should_training_stop = True



        # # Val loss early stopping — stop if val loss increases
        # val_loss = kwargs.get('metrics', {}).get('eval_loss', None)
        # if val_loss is not None:
        #     print(f'📉 Val loss: {val_loss:.6f} (prev: {self.last_val_loss:.6f})')
        #     if val_loss > self.last_val_loss:
        #         print('🛑 Early stopping: val loss increased!')
        #         control.should_training_stop = True
        #     self.last_val_loss = val_loss

        kwargs['metrics']['eval_wer'] = wer   # ✅ pass WER to Trainer

        model.train()
        gc.collect()
        torch.cuda.empty_cache()
        print(f'GPU Memory after eval: {torch.cuda.memory_allocated() / 1e9:.2f} GB / {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

        return control    # ✅ always at the very end

    def on_train_end(self, args, state, control, **kwargs):
        print(f'\nBest chunk-level WER: {self.best_wer:.4f}')


evaluator = ChunkWERCallback(val_data, processor)
print('✅ Chunk-level WER evaluator ready')

class CheckpointCleanupCallback(TrainerCallback):
    """
    Delete old checkpoints BEFORE saving new ones.
    Keeps only 1 checkpoint at a time to save disk space.
    Never deletes best_model.
    """
    def on_step_begin(self, args, state, control, **kwargs):
        # Only check right before saving
        if state.global_step % args.save_steps == 0 and state.global_step > 0:
            checkpoints = sorted(
                [d for d in Path(args.output_dir).glob('checkpoint-*') if d.is_dir()],
                key=lambda x: int(x.name.split('-')[1])
            )
            
            # Delete ALL checkpoints before new one is created
            if checkpoints:
                for old_checkpoint in checkpoints:
                    import shutil
                    shutil.rmtree(old_checkpoint)
                    print(f'🗑️  Deleted checkpoint: {old_checkpoint.name} ({9}GB freed)')

checkpoint_cleaner = CheckpointCleanupCallback()
print('✅ Checkpoint cleanup callback ready')

In [ ]:
wer_metric = evaluate.load('wer')

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {'wer': wer}

## 11. Training Arguments

In [ ]:
os.makedirs(CONFIG['output_dir'], exist_ok=True)

training_args = Seq2SeqTrainingArguments(
    output_dir                  = CONFIG['output_dir'],
    num_train_epochs            = CONFIG['num_epochs'],
    per_device_train_batch_size = CONFIG['batch_size'],
    gradient_accumulation_steps = CONFIG['gradient_accumulation_steps'],
    learning_rate               = CONFIG['learning_rate'],
    warmup_steps                = CONFIG['warmup_steps'],
    weight_decay                = CONFIG['weight_decay'],
    # fp16                        = CONFIG['fp16'],
    # gradient_checkpointing      = CONFIG['gradient_checkpointing'],

    # eval_strategy         = 'steps',
    fp16                        = False,
    bf16                        = False,
    gradient_checkpointing      = CONFIG['gradient_checkpointing'],

    eval_strategy         = 'steps',
    eval_steps                  = CONFIG['eval_steps'],
    save_strategy               = 'steps',
    save_steps                  = CONFIG['save_steps'],
    logging_steps               = CONFIG['logging_steps'],
    save_total_limit            = CONFIG['save_total_limit'],
    # load_best_model_at_end      = True,
    load_best_model_at_end      = False,
    # metric_for_best_model       = CONFIG['metric_for_best_model'],
    # greater_is_better           = CONFIG['greater_is_better'],


    predict_with_generate       = False,
    report_to                   = [],
    push_to_hub                 = False,
    remove_unused_columns       = False,
    label_names                 = ['labels'],
)

print('✅ Training arguments configured')

In [ ]:
# # Early stopping callback
# early_stopping = EarlyStoppingCallback(
#     early_stopping_patience=2,
#     early_stopping_threshold=0.001,
# )
# print('✅ Early stopping callback ready')
# print(f'   Patience: 2 evaluations')
# print(f'   Threshold: 0.001 WER improvement')

## 12. Initialize Trainer

In [ ]:
trainer = Seq2SeqTrainer(
    model         = model,
    args          = training_args,
    train_dataset = train_dataset,
    eval_dataset  = val_dataset,
    data_collator = data_collator,
    tokenizer     = processor.tokenizer,
    #compute_metrics = compute_metrics,
    callbacks     = [evaluator, checkpoint_cleaner],
)

print('✅ Trainer initialized')
print(f'   Train samples : {len(train_dataset)}')
print(f'   Val samples   : {len(val_dataset)}')
print(f'   Eval every    : {CONFIG["eval_steps"]} steps')

In [ ]:
# Clean up old checkpoints to free space
import shutil
checkpoint_dirs = [d for d in Path(CONFIG['output_dir']).glob('checkpoint-*')]
if checkpoint_dirs:
    print(f'Cleaning up {len(checkpoint_dirs)} old checkpoints...')
    for ckpt in checkpoint_dirs:
        shutil.rmtree(ckpt)
    print(f'✅ Freed space from old checkpoints')

## 13. Train 🚀

In [ ]:
print('\n' + '='*60)
print('STARTING TRAINING')
print('='*60)
print(f'  Train chunks : {len(train_dataset)}')
print(f'  Val chunks   : {len(val_dataset)}')
print(f'  Epochs       : {CONFIG["num_epochs"]}')
print(f'  Batch size   : {CONFIG["batch_size"]} × {CONFIG["gradient_accumulation_steps"]} accum')
print(f'  Eval every   : {CONFIG["eval_steps"]} steps (chunk-level WER, fast)')
print('='*60 + '\n')

import gc
gc.collect()
torch.cuda.empty_cache()

print(f'GPU Memory before training: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

train_result = trainer.train()
print('\n✅ Training complete!')

# Load best model back into memory
model = WhisperForConditionalGeneration.from_pretrained(
    './whisper-bengali-finetuned/best_model'
)
model.to(device)
print(f'✅ Best model loaded (WER: {evaluator.best_wer:.4f})')

## 14. Save Final Model

In [ ]:
print('Saving final model...')
model.save_pretrained(CONFIG['output_dir'])
processor.save_pretrained(CONFIG['output_dir'])

print(f'\n✅ Saved to: {CONFIG["output_dir"]}')
print(f'📊 Best chunk-level WER: {evaluator.best_wer:.4f}')

## 15. Quick Sanity Check on Val Sample

In [ ]:
sample = val_data[0]
audio, sr = sf.read(sample['audio_path'])
if audio.ndim > 1:
    audio = audio.mean(axis=1)
audio = audio.astype(np.float32)

inputs = processor.feature_extractor(
    audio, sampling_rate=CONFIG['sampling_rate'], return_tensors='pt'
).input_features.to(device)

if CONFIG['fp16']:
    inputs = inputs.half()

from transformers import GenerationConfig
model.generation_config = GenerationConfig.from_pretrained("openai/whisper-medium")
model.generation_config.language = CONFIG['language']
model.generation_config.task = CONFIG['task']

with torch.no_grad():
    predicted_ids = model.generate(
        inputs,
        language=CONFIG['language'],
        task=CONFIG['task'],
        max_new_tokens=440,
    )

pred_text = processor.tokenizer.batch_decode(predicted_ids, skip_special_tokens=True)[0].strip()
print(f'File        : {sample["file_id"]}')
print(f'Ground truth: {sample["text"][:200]}')
print(f'Prediction  : {pred_text[:200]}')